In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline, BertTokenizer, BertForSequenceClassification
from datetime import timedelta
from tqdm.auto import tqdm

In [2]:
TICKERS = [
    'AXP','AMGN','AAPL','BA','CAT','CSCO','CVX','GS','HD','HON','IBM','INTC','JNJ',
    'KO','JPM','MCD','MMM','MRK','MSFT','NKE','PG','TRV','UNH','CRM','VZ','V','WBA',
    'WMT','DIS','DOW'
]

In [3]:
TRAIN_START_DATE = '2016-01-01'
TRAIN_END_DATE = '2019-01-01'
TRADE_START_DATE = '2019-01-01'
TRADE_END_DATE = '2019-07-01'

In [4]:
train = pd.read_csv('train_data.csv', parse_dates=['date'])
trade = pd.read_csv('trade_data.csv', parse_dates=['date'])

In [ ]:
overall_start_date = pd.to_datetime(TRAIN_START_DATE)
overall_end_date = pd.to_datetime(TRADE_END_DATE)

print(f"Filtering news data for tickers: {TICKERS}")
print(f"Filtering news data between {overall_start_date.date()} and {overall_end_date.date()}")

news = pd.read_csv('analyst_ratings_processed.csv')

news = news[news['stock'].isin(TICKERS)].copy() 

news['date'] = pd.to_datetime(
    news['date'], utc=True, errors='coerce'
).dt.tz_localize(None)
news = news.dropna(subset=['date'])

news = news[
    (news['date'] >= overall_start_date) &
    (news['date'] <= overall_end_date)
].copy()

news = news.rename(columns={'stock':'tic', 'title':'headline'})

news = news.sort_values('date').reset_index(drop=True)

print(f"Final news data shape for sentiment analysis: {news.shape}")

Filtering news data for tickers: ['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']
Filtering news data between 2016-01-01 and 2019-07-01
Final news data shape for sentiment analysis: (12563, 4)


In [ ]:
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone')
try:
    sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0) # Try GPU
except Exception as e:
    sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=-1) # Fallback to CPU

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Device set to use cuda:0


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

In [7]:
label_map = {'POSITIVE': 1, 'NEUTRAL': 0, 'NEGATIVE': -1}

In [ ]:
def score_texts(texts):
    """Compute the average sentiment score for a list of headlines."""
    if not texts: 
        return 0.0 
    try:
        valid_texts = [str(text) for text in texts if isinstance(text, str) and text.strip()]
        if not valid_texts:
            return 0.0

        results = sentiment(valid_texts, batch_size=16)
        scores  = [label_map.get(r['label'].upper(), 0) * r['score'] for r in results]
        return float(np.mean(scores))
    except Exception as e:
        print(f"Error scoring texts: {texts}. Error: {e}")
        return 0.0

In [ ]:
def build_sentiment_dict(df, news_df):
    """Builds a dictionary mapping (tic, date) to sentiment score."""
    sent = {}
    df['date'] = pd.to_datetime(df['date'])
    unique_dates = df[['tic','date']].drop_duplicates()

    news_df['date'] = pd.to_datetime(news_df['date'])

    for tic, current_date in tqdm(unique_dates.itertuples(index=False),
                                   total=len(unique_dates),
                                   desc="Computing sentiment"):

        window_start = current_date - timedelta(days=14)

        mask = (
            (news_df['tic'] == tic) &
            (news_df['date'] > window_start) &
            (news_df['date'] <= current_date)
        )
        headlines = news_df.loc[mask, 'headline'].tolist()

        sent[(tic, current_date)] = score_texts(headlines)

    return sent

train_sent = build_sentiment_dict(train, news) 
trade_sent = build_sentiment_dict(trade, news) 

Computing sentiment:   0%|          | 0/21866 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Computing sentiment:   0%|          | 0/3567 [00:00<?, ?it/s]

In [ ]:
def merge_and_save(df, sent_dict, filename):
    """Merges sentiment scores into the dataframe and saves to CSV."""
    sent_df = pd.DataFrame([
        {'tic': tic, 'date': date, 'sentiment_2w': score}
        for (tic, date), score in sent_dict.items()
    ])
    df['date'] = pd.to_datetime(df['date'])
    sent_df['date'] = pd.to_datetime(sent_df['date'])

    df_enriched = df.merge(sent_df, on=['tic','date'], how='left')

    initial_na_count = df_enriched['sentiment_2w'].isna().sum()
    df_enriched['sentiment_2w'] = df_enriched['sentiment_2w'].fillna(0.0)

    df_enriched.to_csv(filename, index=False)
    print(f"Saved enriched data to {filename}. Shape: {df_enriched.shape}")
    return df_enriched

train_enriched = merge_and_save(train, train_sent, 'train_with_sentiment.csv')
trade_enriched = merge_and_save(trade, trade_sent, 'trade_with_sentiment.csv')

Saved enriched data to train_with_sentiment.csv. Shape: (21866, 20)
Saved enriched data to trade_with_sentiment.csv. Shape: (3567, 20)
